# Dobijanje modela za primenu na servisu

Koristi se dataset preuzet sa sajta Kaggle (winequality-red.csv). Sadrži 11 obeležja (prediktora), koja se odnose na karakteristike vina i ocenu kvaliteta (1-10).

Ovaj problem se pretvara u problem binarne klasifikacije, tako što se ocene (1-5) pretvaraju u labelu 0 (loše vino), a ocene veće ili jednake od 6, se pretvaraju u labelu 1 dobro vino.

Podela skupa je trening/validacioni/test.

Pretraga HPO za originalni model RF, Bajesovskom optimizacijom. Maksimizuje se Precision. Treniranje na trening skupu, procena metrika (Precision, Recall, ROC-AUC) na validacionom skupu i računanje veličine modela.

Ponovna pretraga HPO sa suženim opsegom parametara, Bajesovskom optimizacijom. Maksimizuje se Precision i minimizuje veličina modela, preko faktora alfa. Odabir najboljeg alfa. Treniranje na trening skupu, procena metrika (Precision, Recall, ROC-AUC) na validacionom skupu i računanje veličine modela.

Oba modela se potom treniraju na trening + validacionom skupu i procenjuju na nezavisnom test skupu. Odabir konačnog modela.

## Potrebne biblioteke

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import tempfile
from tqdm.auto import tqdm

## Osnovna analiza dataset-a i pretvaranje labela u 0 i 1

In [7]:
df = pd.read_csv(r"winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [8]:
df.shape

(1599, 12)

In [9]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [10]:
df['quality'].value_counts().sort_index()

quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

## Odvajanje prediktora i ciljne promenjive, provera balansiranosti klasa

In [11]:
prediktori = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
ciljnaprom = ['quality']

X = df[prediktori]
y = (df['quality'] >= 6).astype(int)

In [12]:
print(y.value_counts()) 

quality
1    855
0    744
Name: count, dtype: int64


## Podela na trening/validacioni/test skup (70:15:15)

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")
print(f"Train pos%: {y_train.mean():.2f}  Val pos%: {y_val.mean():.2f}  Test pos%: {y_test.mean():.2f}")

Train: 1119  Val: 240  Test: 240
Train pos%: 0.53  Val pos%: 0.54  Test pos%: 0.53


## Pretraga HPO bibliotekom Optuna. Ispis 5 najboljih modela

In [14]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 200),
        "random_state": 42
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    precision = precision_score(y_val, y_pred)

    return precision

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100)

print("\n TOP 5 TRIALS")
top5 = sorted(study.trials, key=lambda t: t.value, reverse=True)[:5]
for i, trial in enumerate(top5):
    model = RandomForestClassifier(**trial.params, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_proba)

    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
        joblib.dump(model, f.name)
        velicina = os.path.getsize(f.name) / 1024
    os.unlink(f.name)

    print(f"\n#{i+1}  Precision: {precision:.4f}  Recall: {recall:.4f}  ROC-AUC: {roc_auc:.4f}  Velicina: {velicina:.1f} KB")
    print(f"     Parametri: {trial.params}")


 TOP 5 TRIALS

#1  Precision: 0.7891  Recall: 0.7829  ROC-AUC: 0.8390  Velicina: 601.8 KB
     Parametri: {'n_estimators': 141, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_leaf_nodes': 54}

#2  Precision: 0.7891  Recall: 0.7829  ROC-AUC: 0.8386  Velicina: 618.8 KB
     Parametri: {'n_estimators': 145, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_leaf_nodes': 50}

#3  Precision: 0.7891  Recall: 0.7829  ROC-AUC: 0.8390  Velicina: 669.9 KB
     Parametri: {'n_estimators': 157, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_leaf_nodes': 37}

#4  Precision: 0.7891  Recall: 0.7829  ROC-AUC: 0.8390  Velicina: 665.4 KB
     Parametri: {'n_estimators': 156, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_leaf_nodes': 32}

#5  Precision: 0.7891  Recall: 0.7829  ROC-AUC: 0.8387  Velicina: 606.5 KB
     Parametri: {'n_estimators': 142, 'max_depth': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_leaf_no

## Treniranje modela na trening skupu, procena na validacionom sa parametrima iz prethodne pretrage

In [15]:
best_params = {
    'n_estimators': 141,
    'max_depth': 5,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'max_leaf_nodes': 54
}

val_model = RandomForestClassifier(**best_params, random_state=42)
val_model.fit(X_train, y_train)
y_pred_val = val_model.predict(X_val)
y_proba_val = val_model.predict_proba(X_val)[:, 1]

print(f"Precision (val): {precision_score(y_val, y_pred_val):.4f}")
print(f"Recall (val):    {recall_score(y_val, y_pred_val):.4f}")
print(f"ROC-AUC (val):   {roc_auc_score(y_val, y_proba_val):.4f}")

best_model = RandomForestClassifier(**best_params, random_state=42)
best_model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

import tempfile
with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
    joblib.dump(best_model, f.name)
    orig_size = os.path.getsize(f.name) / 1024
os.unlink(f.name)
print(f"Originalni model: {orig_size:.1f} KB")

Precision (val): 0.7891
Recall (val):    0.7829
ROC-AUC (val):   0.8390
Originalni model: 625.5 KB


## Nova, sužena pretraga HPO sa novom funkcijom cilja i opsegom za parametar alfa.

In [16]:
max_velicina = orig_size  #originalni model kao referenca
alphas = [round(a * 0.1, 1) for a in range(0, 11)]  # 0.0, 0.1, 0.2, ..., 1.0

def make_objective(alpha):
    def objective_pruned_size(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 50),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 50),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 40),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.02),
            "random_state": 42
        }
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)
        roc_auc = roc_auc_score(y_val, y_proba)

        with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
            joblib.dump(model, f.name)
            velicina = os.path.getsize(f.name) / 1024
        os.unlink(f.name)

        score = precision - alpha * (velicina / max_velicina)

        trial.set_user_attr("precision", precision)
        trial.set_user_attr("recall", recall)
        trial.set_user_attr("roc_auc", roc_auc)
        trial.set_user_attr("size_kb", velicina)

        return score
    return objective_pruned_size

studies_by_alpha = {}
rezultati = []

for alpha in tqdm(alphas, desc="Alpha pretraga"):
    study_pruned_size = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study_pruned_size.optimize(make_objective(alpha), n_trials=100, show_progress_bar=False)

    best = study_pruned_size.best_trial
    studies_by_alpha[alpha] = study_pruned_size

    rezultati.append({
        "alpha": alpha,
        "precision_val": best.user_attrs["precision"],
        "recall_val": best.user_attrs["recall"],
        "roc_auc_val": best.user_attrs["roc_auc"],
        "size_kb": best.user_attrs["size_kb"],
        "params": best.params
    })

summary_df = pd.DataFrame(rezultati).sort_values("alpha").reset_index(drop=True)
summary_df

Alpha pretraga:   0%|          | 0/11 [00:00<?, ?it/s]

,alpha,precision_val,recall_val,roc_auc_val,size_kb,params
0,0.0,0.793893,0.806202,0.835533,83.180664,"{'n_estimators': 48, 'max_depth': 4, 'min_samp..."
1,0.1,0.793893,0.806202,0.828305,47.868164,"{'n_estimators': 39, 'max_depth': 3, 'min_samp..."
2,0.2,0.793893,0.806202,0.826140,46.774414,"{'n_estimators': 37, 'max_depth': 3, 'min_samp..."
3,0.3,0.795276,0.782946,0.819121,16.305664,"{'n_estimators': 13, 'max_depth': 6, 'min_samp..."
4,0.4,0.792308,0.798450,0.823032,19.586914,"{'n_estimators': 16, 'max_depth': 5, 'min_samp..."
5,0.5,0.793651,0.775194,0.830435,26.774414,"{'n_estimators': 14, 'max_depth': 5, 'min_samp..."
6,0.6,0.786260,0.798450,0.818144,19.899414,"{'n_estimators': 17, 'max_depth': 5, 'min_samp..."
7,0.7,0.798246,0.705426,0.805014,10.368164,"{'n_estimators': 12, 'max_depth': 2, 'min_samp..."
8,0.8,0.795276,0.782946,0.817620,16.305664,"{'n_estimators': 13, 'max_depth': 5, 'min_samp..."
9,0.9,0.791304,0.705426,0.801732,10.211914,"{'n_estimators': 12, 'max_depth': 2, 'min_samp..."


## Odabrano alfa koje je zadržalo vrednosti metrika uz minimalan pad, a što veće smanjenje veličine. Treniranje na trening skupu, procena na validacionom

In [17]:
#Hardkodovano iz studies_by_alpha[0.4].best_params (alpha-sweep gore, selekcija iskljucivo po val metrikama/velicini)
#Test skup ovde namerno NE diramo - sva evaluacija na test je premestena u finalnu sekciju na kraju notebook-a.
best_pruned_params = {
    'n_estimators': 16,
    'max_depth': 5,
    'min_samples_leaf': 37,
    'max_features': 'sqrt',
    'max_leaf_nodes': 11,
    'ccp_alpha': 0.0105694743860919
}

val_model_pruned = RandomForestClassifier(**best_pruned_params, random_state=42)
val_model_pruned.fit(X_train, y_train)
y_pred_val = val_model_pruned.predict(X_val)
y_proba_val = val_model_pruned.predict_proba(X_val)[:, 1]

with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
    joblib.dump(val_model_pruned, f.name)
    val_pruned_size = os.path.getsize(f.name) / 1024
os.unlink(f.name)

print(f"Precision (val): {precision_score(y_val, y_pred_val):.4f}")
print(f"Recall (val):    {recall_score(y_val, y_pred_val):.4f}")
print(f"ROC-AUC (val):   {roc_auc_score(y_val, y_proba_val):.4f}")
print(f"Pruned model:    {val_pruned_size:.1f} KB")

final_pruned_model = RandomForestClassifier(**best_pruned_params, random_state=42)
final_pruned_model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

joblib.dump(final_pruned_model, "final_model.joblib")
pruned_size = os.path.getsize("final_model.joblib") / 1024

Precision (val): 0.7923
Recall (val):    0.7984
ROC-AUC (val):   0.8230
Pruned model:    19.6 KB


## Finalna procena na test skupu, za oba modela. Poređenje.

In [18]:
print("ORIGINALNI MODEL NA TEST SKUPU")
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

print("\nPRUNED MODEL NA TEST SKUPU")
y_pred = final_pruned_model.predict(X_test)
y_proba = final_pruned_model.predict_proba(X_test)[:, 1]
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall {recall_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

print("\nVELICINA")
print(f"Originalni: {orig_size:.1f} KB")
print(f"Pruned: {pruned_size:.1f} KB")
print(f"Smanjenje: {(1 - pruned_size/orig_size)*100:.1f}%")


ORIGINALNI MODEL NA TEST SKUPU
Precision: 0.7458
Recall: 0.6875
ROC-AUC: 0.8363

PRUNED MODEL NA TEST SKUPU
Precision: 0.7627
Recall 0.7031
ROC-AUC: 0.8068

VELICINA
Originalni: 625.5 KB
Pruned: 19.9 KB
Smanjenje: 96.8%


## Odabran je prunnovan model zato što pokazuje poboljšanje (Precison i Recall) ili minimalan pad (ROC-AUC) metrika, uz veliku kompresiju (96.8%)

In [19]:
import os
print(os.getcwd())
print(os.path.exists("final_model.joblib"))

c:\Users\neman\wine_api
True


## Korelaciona analiza, kasnije korišćena (pogledati Prezentacija.pdf)

In [20]:
df[['alcohol', 'density', 'sulphates', 'pH']].corrwith(y)


alcohol      0.434751
density     -0.159110
sulphates    0.218072
pH          -0.003264
dtype: float64